In [ ]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.python.keras import regularizers
import statistics as st
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import classification_report
from sklearn.utils import class_weight
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import keras
from keras import backend as K
from keras.models import Sequential, model_from_yaml
from keras.layers import Dense, Dropout, Bidirectional, LSTM
from mpl_toolkits import mplot3d

### Dataset files 

In [ ]:
# List of training, validation, and test X_files
X_files = ['TrainingData/subject_001_01__x.csv', 'TrainingData/subject_001_02__x.csv', 
           'TrainingData/subject_001_03__x.csv', 'TrainingData/subject_001_04__x.csv', 
           'TrainingData/subject_001_05__x.csv', 'TrainingData/subject_001_06__x.csv', 
           'TrainingData/subject_001_07__x.csv', 'TrainingData/subject_002_02__x.csv', 
           'TrainingData/subject_002_03__x.csv', 'TrainingData/subject_002_04__x.csv', 
           'TrainingData/subject_002_05__x.csv', 'TrainingData/subject_003_01__x.csv', 
           'TrainingData/subject_003_02__x.csv', 'TrainingData/subject_003_03__x.csv', 
           'TrainingData/subject_004_01__x.csv', 'TrainingData/subject_004_02__x.csv', 
           'TrainingData/subject_005_01__x.csv', 'TrainingData/subject_005_02__x.csv', 
           'TrainingData/subject_005_03__x.csv', 'TrainingData/subject_006_01__x.csv', 
           'TrainingData/subject_006_02__x.csv', 'TrainingData/subject_007_02__x.csv', 
           'TrainingData/subject_007_03__x.csv', 'TrainingData/subject_007_04__x.csv',
           'TrainingData/subject_008_01__x.csv']

val_X_files = ['TrainingData/subject_002_01__x.csv', 'TrainingData/subject_001_08__x.csv']
test_X_files = ['TrainingData/subject_006_03__x.csv', 'TrainingData/subject_007_01__x.csv']

# List of training, validation, and test y_files
y_files = ['TrainingData/subject_001_01__y.csv', 'TrainingData/subject_001_02__y.csv', 
           'TrainingData/subject_001_03__y.csv', 'TrainingData/subject_001_04__y.csv', 
           'TrainingData/subject_001_05__y.csv', 'TrainingData/subject_001_06__y.csv', 
           'TrainingData/subject_001_07__y.csv', 'TrainingData/subject_002_02__y.csv',
           'TrainingData/subject_002_03__y.csv', 'TrainingData/subject_002_04__y.csv', 
           'TrainingData/subject_002_05__y.csv', 'TrainingData/subject_003_01__y.csv', 
           'TrainingData/subject_003_02__y.csv', 'TrainingData/subject_003_03__y.csv', 
           'TrainingData/subject_004_01__y.csv', 'TrainingData/subject_004_02__y.csv', 
           'TrainingData/subject_005_01__y.csv', 'TrainingData/subject_005_02__y.csv', 
           'TrainingData/subject_005_03__y.csv', 'TrainingData/subject_006_01__y.csv', 
           'TrainingData/subject_006_02__y.csv', 'TrainingData/subject_007_02__y.csv', 
           'TrainingData/subject_007_03__y.csv', 'TrainingData/subject_007_04__y.csv',
           'TrainingData/subject_008_01__y.csv']

val_y_files = ['TrainingData/subject_002_01__y.csv', 'TrainingData/subject_001_08__y.csv']
test_y_files = ['TrainingData/subject_006_03__y.csv', 'TrainingData/subject_007_01__y.csv']

## Visuvalising the Data 

In [ ]:
l=[]

for i in range(len(y_files)):
    df_X = pd.read_csv(X_files[i],names =['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z'])
    df_y = pd.read_csv(y_files[i],names =['label'])
    e_labels = []
    for label in df_y.iterrows():
        e_labels += [label[1][0]] * 4
    df_y = pd.DataFrame(e_labels)
    df_y.columns = ['label']
    df_X = df_X.join(df_y, how='right')
    l.append(df_X)
df_f = pd.concat(l1, axis=0, ignore_index=True)
df_f

In [ ]:
walking = df_f[df_f.label==0].count()[0]
downstairs = df_f[df_f.label==1].count()[0]
upstairs = df_f[df_f.label==2].count()[0]
grass = df_f[df_f.label==3].count()[0]
print("Walking =",walking,"\nDownstairs = ",downstairs,"\nUpstairs = ",upstairs,"\nGrass = ",grass)

In [ ]:
text = ["standing / walking", "going downstairs","going upstairs", "walking on grass"]
label = [walking, downstairs,upstairs, grass]
plt.style.use('seaborn')
plt.figure(figsize=(8,6),dpi=100)
for bar in range(0,4):
    plt.bar(text[bar],label[bar])

In [ ]:
plt.subplot(2,2,1)
plt.plot(df_f[df_f.label==0][['acc_x', 'acc_y', 'acc_z']])
plt.title('standing / walking')
plt.subplot(2,2,2)
plt.plot(df_f[df_f.label==1][['acc_x', 'acc_y', 'acc_z']])
plt.title('going downstairs')
plt.legend(['acc_x','acc_y','acc_z'])
plt.subplot(2,2,3)
plt.plot(df_f[df_f.label==2][['acc_x', 'acc_y', 'acc_z']])
plt.title('going upstairs')
plt.subplot(2,2,4)
plt.plot(df_f[df_f.label==3][['acc_x', 'acc_y', 'acc_z']])
plt.title('walking on grass')
plt.show()

In [ ]:

plt.subplot(2,2,1)
plt.plot(df_f[df_f.label==0][['gyro_x', 'gyro_y', 'gyro_z']])
plt.title('standing / walking')
plt.legend(['gyro_x', 'gyro_y', 'gyro_z'],loc='lower right')
plt.subplot(2,2,2)
plt.plot(df_f[df_f.label==1][['gyro_x', 'gyro_y', 'gyro_z']])
plt.title('going downstairs')
plt.subplot(2,2,3)
plt.plot(df_f[df_f.label==2][['gyro_x', 'gyro_y', 'gyro_z']])
plt.title('going upstairs')
plt.subplot(2,2,4)
plt.plot(df_f[df_f.label==3][['gyro_x', 'gyro_y', 'gyro_z']])
plt.title('walking on grass')
plt.show()

In [ ]:
a_x=df_f[df_f.label==1]['acc_x']
a_y=df_f[df_f.label==1]['acc_y']
a_z=df_f[df_f.label==1]['acc_z']


fig = plt.figure(figsize = (10, 7))
ax = plt.axes(projection ="3d")
ax.scatter3D(a_x,a_y,a_z,)
plt.title("Standing / Walking")
plt.show()

# preprocessing

In [ ]:
def upsampling(X_file, y_file):
    df_X = pd.read_csv(X_file)
    df_y = pd.read_csv(y_file)
    
    extrapolated_labels = []
    for label in df_y.iterrows():
        extrapolated_labels += [label[1][0]] * 4
    
    extrapolated_labels_df = pd.DataFrame(extrapolated_labels)
    difference = df_X.shape[0] - extrapolated_labels_df.shape[0]
    df_X = df_X.iloc[:-difference,:]
    
    return df_X, extrapolated_labels_df


def mode_labels(X, y, time_step, step_size):
    X_values = []
    y_values = []
    for i in range(0, len(X) - time_step, step_size):
        value = X.iloc[i:(i + time_step)].values
        X_values.append(value)
        labels = y.iloc[i:(i + time_step)]
        y_values.append(stats.mode(labels,keepdims=True)[0][0])
    return np.array(X_values),np.array(y_values).reshape(-1, 1)


def create_series_data(X_files, y_files, time_step, step_size):
    aggregate_X = []
    aggregate_y = []
    for i in range(len(y_files)):
        X, y = upsampling(X_files[i], y_files[i])
        X, y = mode_labels(X, y, time_step, step_size)
        aggregate_X.append(X)
        aggregate_y.append(y)
    return np.concatenate(aggregate_X), np.concatenate(aggregate_y)



In [ ]:
training_X, training_y = create_series_data(X_files, y_files, 30, 1)
val_X, val_y = create_series_data(val_X_files, val_y_files, 30, 1)
test_X, test_y = create_series_data(test_X_files, test_y_files, 30, 1)

print(training_X.shape, training_y.shape)
print(val_X.shape, val_y.shape)
print(test_X.shape, test_y.shape)

In [ ]:
# Saving the numpy
np.save('processed_data_1/training_X.npy', training_X)
np.save('processed_data_1/training_y.npy', training_y)
np.save('processed_data_1/val_X.npy', val_X)
np.save('processed_data_1/val_y.npy', val_y)
np.save('processed_data_1/test_X.npy', test_X)
np.save('processed_data_1/test_y.npy', test_y)

In [ ]:
# Loading the numpy
training_X = np.load('processed_data_1/training_X.npy')
training_y = np.load('processed_data_1/training_y.npy')
val_X = np.load('processed_data_1/val_X.npy')
val_y = np.load('processed_data_1/val_y.npy')
test_X = np.load('processed_data_1/test_X.npy')
test_y = np.load('processed_data_1/test_y.npy')

In [ ]:
def get_label_weights(training_y):
    weights = class_weight.compute_class_weight('balanced', classes=np.unique(training_y), y=training_y.ravel())
    label_weights = {i:weights[i] for i in range(len(weights))}
    return label_weights

def one_hot_encoding(labels):
    encoder = OneHotEncoder(handle_unknown = 'ignore', sparse_output= False)
    encoder = encoder.fit(labels)
    training_y_encoded = encoder.transform(labels)
    return training_y_encoded

In [ ]:
label_weights = get_label_weights(training_y)
print(label_weights)

training_y_encoded = one_hot_encoding(training_y)
val_y_encoded = one_hot_encoding(val_y)
test_y_encoded = one_hot_encoding(test_y)
print(training_y_encoded.shape, val_y_encoded.shape, test_y_encoded.shape)

In [ ]:
def define_BiLSTM_model(training_X, training_y_encoded):
    n_timesteps, n_features, n_outputs = training_X.shape[1], training_X.shape[2], training_y_encoded.shape[1]
    model = Sequential()
    model.add(Bidirectional(LSTM(units = 125), input_shape = (n_timesteps, n_features)))
    model.add(Dropout(rate = 0.5))
    model.add(Dense(units = 125, activation = 'relu'))
    model.add(Dense(n_outputs, activation = 'softmax'))
    model.compile(loss = 'categorical_crossentropy', optimizer = 'adam',metrics=['accuracy'])
    return model


In [ ]:
model = define_BiLSTM_model(training_X, training_y_encoded)
model.summary()

In [ ]:
history = model.fit(training_X, training_y_encoded, epochs = 5, batch_size = 64,
                   validation_data = (val_X, val_y_encoded), class_weight = label_weights,
                   verbose = 1, shuffle = True)

In [ ]:
def plot_history(history):
	# Plot loss
    plt.title('Loss')
    plt.plot(history.history['loss'], color='blue', label='train')
    plt.plot(history.history['val_loss'], color='red', label='test')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'])
    plt.show()
    
    # Plot accuracy
    plt.title('Accuracy')
    plt.plot(history.history['accuracy'], color='blue', label='train')
    plt.plot(history.history['val_accuracy'], color='red', label='test')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'])
    plt.show()



In [ ]:
model.evaluate(test_X, test_y_encoded)

In [ ]:
plot_history(history)

# saving the model

In [ ]:
model.save('lstm_weights.h5')

# loading the model

In [ ]:
model=keras.models.load_model('lstm_weights.h5')

In [ ]:
model.evaluate(test_X, test_y_encoded)

In [ ]:
predicted_data = model.predict(test_X, batch_size = 64, verbose = 1)
y_test_bool = np.argmax(predicted_data, axis = 1)
print(classification_report(test_y, y_test_bool))

In [ ]:
test_files = ['TestData/subject_009_01__x.csv']

prediction_files = ['subject_009_01__y_prediction.csv']

In [ ]:
def create_test_dataset(X):
        X_values = []
        for i in range(0, len(X)):
                t=[]
                value = X.iloc[i].values
                t.append(value)
                X_values.append(t)        
        return np.array(X_values)

In [ ]:
for i in range(len(test_files)):
    input_data = pd.read_csv(test_files[i])
    df = input_data
    X_test = create_test_dataset(df)
    y_test = model.predict(X_test, verbose = 1)
    y_test_bool = np.argmax(y_test, axis = 1)
    y_series = pd.Series(y_test_bool)
    y_series.to_csv("predictions/" + prediction_files[i])

In [ ]:
y_series